# Notebook 01: Sentinel Preprocessing

**Outputs:** Cloud-free Sentinel-2 (dry + wet season) and Sentinel-1 (VV, VH) composites for all five countries, exported to GEE Cloud Storage as GeoTIFFs.

**Estimated runtime:** GEE export tasks run asynchronously; submission takes ~5 minutes, actual export 1-4 hours depending on GEE queue.

---

In [ ]:
import ee, geemap, json

# Load project config
with open('../data/config.json') as f:
    CONFIG = json.load(f)

ee.Initialize(project=CONFIG['gee_project'])
print('GEE initialized.')

## 1.1 Define Country Geometries

In [ ]:
# Load LSIB country boundaries from GEE
lsib = ee.FeatureCollection('USDOS/LSIB_SIMPLE/2017')

country_geometries = {}
for country in CONFIG['countries']:
    geom = lsib.filter(ee.Filter.eq('country_na', country)).geometry()
    country_geometries[country] = geom
    print(f'{country}: geometry loaded')

# Full study extent (union of all five)
study_area = ee.FeatureCollection(
    [ee.Feature(g) for g in country_geometries.values()]
).geometry().dissolve()

## 1.2 Define Seasonal Windows

East Africa has two main seasons relevant to vegetation-settlement contrast:
- **Dry season:** June–September (vegetation senescent, settlements more spectrally distinct)
- **Wet season:** November–April (active vegetation, thatch roofs may blend with canopy)

Both composites are generated to exploit temporal contrast in Stage 2.

In [ ]:
SEASONS = {
    'dry':  {'start': '2022-06-01', 'end': '2022-09-30'},
    'wet':  {'start': '2022-11-01', 'end': '2023-04-30'},
}

print('Seasonal windows defined:')
for season, dates in SEASONS.items():
    print(f'  {season}: {dates["start"]} to {dates["end"]}')

## 1.3 Cloud Masking Function (s2cloudless + SCL)

In [ ]:
def mask_s2_clouds(image):
    """
    Mask clouds and cloud shadows using the Sentinel-2 Scene Classification Layer (SCL).
    SCL classes masked out: 3 (cloud shadow), 7 (unclassified), 8 (cloud medium prob),
    9 (cloud high prob), 10 (thin cirrus).
    """
    scl = image.select('SCL')
    # Keep only: 4=vegetation, 5=bare soil, 6=water, 11=snow/ice
    # Mask out clouds, shadows, cirrus
    cloud_mask = (
        scl.neq(3)   # cloud shadow
        .And(scl.neq(7))  # unclassified
        .And(scl.neq(8))  # cloud medium
        .And(scl.neq(9))  # cloud high
        .And(scl.neq(10)) # thin cirrus
    )
    return image.updateMask(cloud_mask).divide(10000)  # scale to surface reflectance

## 1.4 Generate Sentinel-2 Composites

In [ ]:
def make_s2_composite(geometry, start_date, end_date, bands, cloud_pct=20):
    """
    Generate a cloud-free Sentinel-2 median composite for a given geometry and date range.
    Returns a single multi-band image at 10m resolution.
    """
    collection = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(geometry)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_pct))
        .map(mask_s2_clouds)
        .select(bands)
    )
    composite = collection.median().clip(geometry)
    n_images = collection.size().getInfo()
    print(f'  S2 composite: {n_images} scenes used')
    return composite


s2_composites = {}
for country, geom in country_geometries.items():
    s2_composites[country] = {}
    for season, dates in SEASONS.items():
        print(f'Building S2 {season} composite for {country}...')
        composite = make_s2_composite(
            geom,
            dates['start'],
            dates['end'],
            CONFIG['s2_bands'],
            CONFIG['s2_cloud_threshold_pct']
        )
        s2_composites[country][season] = composite

print('\nAll S2 composites built.')

## 1.5 Generate Sentinel-1 Composites

In [ ]:
def make_s1_composite(geometry, start_date, end_date):
    """
    Generate a Sentinel-1 IW GRD median composite (VV, VH) for a given geometry.
    Uses descending orbit pass for consistency across East Africa.
    Returns log-scaled backscatter values.
    """
    collection = (
        ee.ImageCollection('COPERNICUS/S1_GRD')
        .filterBounds(geometry)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.eq('instrumentMode', 'IW'))
        .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
        .select(['VV', 'VH'])
    )
    composite = collection.median().clip(geometry)
    n_images = collection.size().getInfo()
    print(f'  S1 composite: {n_images} scenes used')
    return composite


s1_composites = {}
for country, geom in country_geometries.items():
    s1_composites[country] = {}
    for season, dates in SEASONS.items():
        print(f'Building S1 {season} composite for {country}...')
        s1_composites[country][season] = make_s1_composite(
            geom, dates['start'], dates['end']
        )

print('\nAll S1 composites built.')

## 1.6 Stack Input Channels

For each country and season, stack S2 (4 bands) + S1 (2 bands) into a single 6-band image.
A temporal difference channel (dry - wet NDVI) is computed as the 7th and 8th channels.

In [ ]:
def compute_ndvi(s2_image):
    """Compute NDVI from Sentinel-2 B08 (NIR) and B04 (Red)."""
    return s2_image.normalizedDifference(['B08', 'B04']).rename('NDVI')


stacked_inputs = {}
for country in CONFIG['countries']:
    s2_dry = s2_composites[country]['dry']
    s2_wet = s2_composites[country]['wet']
    s1_dry = s1_composites[country]['dry']
    s1_wet = s1_composites[country]['wet']

    ndvi_dry = compute_ndvi(s2_dry).rename('NDVI_dry')
    ndvi_wet = compute_ndvi(s2_wet).rename('NDVI_wet')
    ndvi_diff = ndvi_dry.subtract(ndvi_wet).rename('NDVI_diff')
    vv_diff = s1_dry.select('VV').subtract(s1_wet.select('VV')).rename('VV_diff')

    # Final 8-band stack:
    # B02, B03, B04, B08 (dry season S2)
    # VV, VH (dry season S1)
    # NDVI_diff (dry - wet)
    # VV_diff (dry - wet)
    stacked = (
        s2_dry
        .addBands(s1_dry)
        .addBands(ndvi_diff)
        .addBands(vv_diff)
    )
    stacked_inputs[country] = stacked
    print(f'{country}: 8-band stack built — bands: {stacked.bandNames().getInfo()}')

print('\nAll input stacks ready for export.')

## 1.7 Export Composites to GEE Cloud Storage

Exports run asynchronously. Check task status at [https://code.earthengine.google.com/tasks](https://code.earthengine.google.com/tasks).  
Do not proceed to Notebook 02 until all export tasks show **COMPLETED**.

In [ ]:
export_tasks = []

for country in CONFIG['countries']:
    task = ee.batch.Export.image.toDrive(
        image=stacked_inputs[country],
        description=f'settlement_input_{country.lower()}',
        folder=CONFIG['gee_export_folder'],
        fileNamePrefix=f'settlement_input_{country.lower()}',
        scale=CONFIG['composite_resolution_m'],
        region=country_geometries[country],
        maxPixels=1e13,
        fileFormat='GeoTIFF',
        formatOptions={'cloudOptimized': True}
    )
    task.start()
    export_tasks.append((country, task))
    print(f'Export task submitted: {country} — task ID: {task.id}')

print(f'\n{len(export_tasks)} export tasks submitted.')
print('Monitor at: https://code.earthengine.google.com/tasks')

## 1.8 Monitor Export Task Status

In [ ]:
import time

# Run this cell periodically to check task status
for country, task in export_tasks:
    status = task.status()
    state = status['state']
    print(f'{country}: {state}')

# All tasks must show COMPLETED before proceeding to Notebook 02

## 1.9 Apply GHS-SMOD Stratification

Label each 500m cell in the study area as rural, peri-urban, or urban using GHS-SMOD 2020.  
This stratification is used in Notebook 02 for pseudo-label sampling.

In [ ]:
smod = ee.Image('JRC/GHSL/P2023A/GHS_SMOD/2020').select('smod_code')

# Reclassify SMOD to three strata
# Rural: classes 10-12, Peri-urban: 21-22, Urban: 23-30
rural_mask    = smod.gte(10).And(smod.lte(12))
periurban_mask = smod.gte(21).And(smod.lte(22))
urban_mask    = smod.gte(23).And(smod.lte(30))

# Encode: 1=rural, 2=peri-urban, 3=urban
stratum = (
    rural_mask.multiply(1)
    .add(periurban_mask.multiply(2))
    .add(urban_mask.multiply(3))
    .rename('stratum')
    .clip(study_area)
)

# Export stratification raster
stratum_task = ee.batch.Export.image.toDrive(
    image=stratum,
    description='smod_stratum_5country',
    folder=CONFIG['gee_export_folder'],
    fileNamePrefix='smod_stratum_5country',
    scale=CONFIG['confidence_resolution_m'],  # 500m
    region=study_area,
    maxPixels=1e10,
    fileFormat='GeoTIFF',
    formatOptions={'cloudOptimized': True}
)
stratum_task.start()
print(f'SMOD stratification export submitted — task ID: {stratum_task.id}')

---
## Stage 1 Complete

When all export tasks show **COMPLETED**, proceed to **Notebook 02: Model Fine-Tuning**.

**Files produced in GEE Drive folder `east_africa_settlement`:**
- `settlement_input_kenya.tif` (8 bands, 10m)
- `settlement_input_tanzania.tif`
- `settlement_input_uganda.tif`
- `settlement_input_rwanda.tif`
- `settlement_input_burundi.tif`
- `smod_stratum_5country.tif` (500m, 3-class stratum)